# Transfer Learning CIFAR10

* Train a simple convnet on the CIFAR dataset the first 5 output classes [0..4].
* Freeze convolutional layers and fine-tune dense layers for the last 5 ouput classes [5..9].


In [1]:
import numpy as np
import pandas as pd
import keras

# Initialize the random number generator
import random
random.seed(0)

import warnings
warnings.filterwarnings("ignore")

Using TensorFlow backend.


In [2]:
from keras.backend import backend
from keras.datasets import cifar10

# the data, shuffled and split between train and test sets
(x_train,y_train),(x_test,y_test)=keras.datasets.cifar10.load_data()

170500096/170498071 [==============================] - 11s 0us/step


In [3]:
index_0to4 = np.where(y_train<5)[0]
x_train_0to4 = x_train[index_0to4]
y_train_0to4 = y_train[index_0to4]
print(x_train_0to4.shape,y_train_0to4.shape)

(25000, 32, 32, 3) (25000, 1)


In [4]:
index_5to9 = np.where(y_train>4)[0]
x_train_5to9 = x_train[index_5to9]
y_train_5to9 = y_train[index_5to9]
print(x_train_5to9.shape,y_train_5to9.shape)

(25000, 32, 32, 3) (25000, 1)


In [5]:
index_test_0to4 = np.where(y_test<5)[0]
x_test_0to4 = x_test[index_test_0to4]
y_test_0to4 = y_test[index_test_0to4]
print(x_test_0to4.shape,y_test_0to4.shape)

(5000, 32, 32, 3) (5000, 1)


In [6]:
index_test_5to9 = np.where(y_test>4)[0]
x_test_5to9 = x_test[index_test_5to9]
y_test_5to9 = y_test[index_test_5to9]
print(x_test_5to9.shape,y_test_5to9.shape)

(5000, 32, 32, 3) (5000, 1)


### 2. Use One-hot encoding to divide y_train and y_test into required no of output classes

In [0]:
y_train_0to4_enc = keras.utils.to_categorical(y_train_0to4)
y_train_5to9_enc = keras.utils.to_categorical(y_train_5to9)[:,5:]
y_test_0to4_enc = keras.utils.to_categorical(y_test_0to4)
y_test_5to9_enc = keras.utils.to_categorical(y_test_5to9)[:,5:]

### 3. Build a sequential neural network model which can classify the classes 0 to 4 of CIFAR10 dataset with at least 80% accuracy on test data

In [0]:
from keras.models import Sequential
from keras.layers import Dense,Conv2D,Flatten,MaxPool2D,Dropout
from keras import callbacks

### 4. In the model which was built above (for classification of classes 0-4 in CIFAR10), make only the dense layers to be trainable and conv layers to be non-trainable

In [9]:
# Initializing Sequential Model
model_0to4 = Sequential()

# Addition of 1st convolution layer
model_0to4.add(Conv2D(24,(3,3),input_shape=(32,32,3),activation='relu'))

# Addition of MaxPooling layer
model_0to4.add(MaxPool2D(pool_size=(2,2)))

# Addition of 2nd convolution layer
model_0to4.add(Conv2D(256,(3,3),activation='relu'))

# Addition of 3rd Covolution layer
model_0to4.add(Conv2D(512,(3,3),activation = 'relu'))

# Addition of MaxPooling layer
model_0to4.add(MaxPool2D(pool_size=(2,2)))

# Flatten the results for dense layer
model_0to4.add(Flatten())

# Adding 1st Dense layer
model_0to4.add(Dense(500,activation='sigmoid'))

# Adding 15% dropout layer
model_0to4.add(Dropout(0.35))


# Adding 3rd Dense layer
model_0to4.add(Dense(150,activation='sigmoid'))

# Adding 4th Dense layer
model_0to4.add(Dense(50,activation='sigmoid'))

#Adding output layer
model_0to4.add(Dense(5,activation='softmax'))






Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


In [10]:
sgd_opt = keras.optimizers.SGD(lr=0.02)
model_0to4.compile(loss='categorical_crossentropy',optimizer=sgd_opt,metrics=['accuracy'])

In [12]:
# Monitoring val_acc and stop training when it stops improving
callback_list = [callbacks.EarlyStopping(monitor='val_acc',patience=5,verbose=1,mode='auto')]

model_0to4.fit(x_train_0to4,y_train_0to4_enc,batch_size=32,epochs=50,callbacks=callback_list,
               validation_data=(x_test_0to4,y_test_0to4_enc))

Train on 25000 samples, validate on 5000 samples
Epoch 1/50
25000/25000 [==============================] - 19s 764us/step - loss: 0.6915 - acc: 0.7383 - val_loss: 1.1423 - val_acc: 0.5346
Epoch 2/50
25000/25000 [==============================] - 19s 761us/step - loss: 0.6076 - acc: 0.7760 - val_loss: 1.8091 - val_acc: 0.4342
Epoch 3/50
25000/25000 [==============================] - 19s 759us/step - loss: 0.5318 - acc: 0.8076 - val_loss: 0.8868 - val_acc: 0.6800
Epoch 4/50
25000/25000 [==============================] - 19s 757us/step - loss: 0.4622 - acc: 0.8329 - val_loss: 0.6159 - val_acc: 0.7710
Epoch 5/50
25000/25000 [==============================] - 19s 764us/step - loss: 0.3909 - acc: 0.8608 - val_loss: 0.7421 - val_acc: 0.7344
Epoch 6/50
25000/25000 [==============================] - 19s 768us/step - loss: 0.3273 - acc: 0.8856 - val_loss: 0.6197 - val_acc: 0.7806
Epoch 7/50
25000/25000 [==============================] - 19s 763us/step - loss: 0.2667 - acc: 0.9096 - val_loss: 0.7

**In the model which was built above (for classification of classes 0-4 in CIFAR10), make only the dense layers to be trainable and conv layers to be non-trainable**

In [13]:
model_0to4.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 30, 30, 24)        672       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 15, 15, 24)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 13, 13, 256)       55552     
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 11, 11, 512)       1180160   
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 5, 5, 512)         0         
_________________________________________________________________
flatten_1 (Flatten)          (None, 12800)             0         
_________________________________________________________________
dense_1 (Dense)              (None, 500)              

In [0]:
for layer in model_0to4.layers:
  if(type(layer) == Conv2D) :
    layer.trainable=False

In [15]:
model_0to4.summary()

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 30, 30, 24)        672       
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 15, 15, 24)        0         
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 13, 13, 256)       55552     
_________________________________________________________________
conv2d_3 (Conv2D)            (None, 11, 11, 512)       1180160   
_________________________________________________________________
max_pooling2d_2 (MaxPooling2 (None, 5, 5, 512)         0         
_________________________________________________________________
flatten_1 (Flatten)          (None, 12800)             0         
_________________________________________________________________
dense_1 (Dense)              (None, 500)              

### 5. Utilize the the model trained on CIFAR 10 (classes 0 to 4) to classify the classes 5 to 9 of CIFAR 10  (Use Transfer Learning) <br>
Achieve an accuracy of more than 85% on test data

In [16]:
# Monitoring val_acc and stop training when it stops improving
callback_list = [callbacks.EarlyStopping(monitor='val_acc',patience=5,verbose=1,mode='auto')]

model_0to4.fit(x_train_5to9,y_train_5to9_enc,batch_size=32,epochs=50,callbacks=callback_list,
               validation_data=(x_test_5to9,y_test_5to9_enc))

Train on 25000 samples, validate on 5000 samples
Epoch 1/50
25000/25000 [==============================] - 19s 763us/step - loss: 0.9246 - acc: 0.6614 - val_loss: 0.5408 - val_acc: 0.8076
Epoch 2/50
25000/25000 [==============================] - 19s 763us/step - loss: 0.5410 - acc: 0.7995 - val_loss: 0.5465 - val_acc: 0.8066
Epoch 3/50
25000/25000 [==============================] - 19s 757us/step - loss: 0.4186 - acc: 0.8478 - val_loss: 0.5440 - val_acc: 0.8034
Epoch 4/50
25000/25000 [==============================] - 19s 758us/step - loss: 0.3359 - acc: 0.8772 - val_loss: 0.4581 - val_acc: 0.8404
Epoch 5/50
25000/25000 [==============================] - 19s 758us/step - loss: 0.2586 - acc: 0.9082 - val_loss: 0.4421 - val_acc: 0.8504
Epoch 6/50
25000/25000 [==============================] - 19s 763us/step - loss: 0.1938 - acc: 0.9318 - val_loss: 0.5219 - val_acc: 0.8250
Epoch 7/50
25000/25000 [==============================] - 19s 759us/step - loss: 0.1344 - acc: 0.9554 - val_loss: 0.3

# Text classification using TF-IDF

### 6. Load the dataset from sklearn.datasets

In [0]:
from sklearn.datasets import fetch_20newsgroups

In [0]:
categories = ['alt.atheism', 'soc.religion.christian', 'comp.graphics', 'sci.med']

### 7. Training data

In [19]:
twenty_train = fetch_20newsgroups(subset='train', categories=categories, shuffle=True, random_state=42)

### 8. Test data

In [0]:
twenty_test = fetch_20newsgroups(subset='test', categories=categories, shuffle=True, random_state=42)

###  a.  You can access the values for the target variable using .target attribute 
###  b. You can access the name of the class in the target variable with .target_names


In [21]:
twenty_train.target

array([1, 1, 3, ..., 2, 2, 2])

In [22]:
twenty_train.target_names

['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']

In [23]:
twenty_train.data[0:5]

['From: sd345@city.ac.uk (Michael Collier)\nSubject: Converting images to HP LaserJet III?\nNntp-Posting-Host: hampton\nOrganization: The City University\nLines: 14\n\nDoes anyone know of a good way (standard PC application/PD utility) to\nconvert tif/img/tga files into LaserJet III format.  We would also like to\ndo the same, converting to HPGL (HP plotter) files.\n\nPlease email any response.\n\nIs this the correct group?\n\nThanks in advance.  Michael.\n-- \nMichael Collier (Programmer)                 The Computer Unit,\nEmail: M.P.Collier@uk.ac.city                The City University,\nTel: 071 477-8000 x3769                      London,\nFax: 071 477-8565                            EC1V 0HB.\n',
 "From: ani@ms.uky.edu (Aniruddha B. Deglurkar)\nSubject: help: Splitting a trimming region along a mesh \nOrganization: University Of Kentucky, Dept. of Math Sciences\nLines: 28\n\n\n\n\tHi,\n\n\tI have a problem, I hope some of the 'gurus' can help me solve.\n\n\tBackground of the probl

### 9.  Now with dependent and independent data available for both train and test datasets, using TfidfVectorizer fit and transform the training data and test data and get the tfidf features for both

In [0]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [0]:
vectorizer = TfidfVectorizer()
train_X = vectorizer.fit_transform(twenty_train.data).toarray()
train_Y = twenty_train.target
test_X = vectorizer.transform(twenty_test.data).toarray()
test_Y = twenty_test.target

### 10. Use logisticRegression with tfidf features as input and targets as output and train the model and report the train and test accuracy score

In [0]:
from sklearn.linear_model import LogisticRegression

In [27]:
vector_model = LogisticRegression()
vector_model.fit(train_X,train_Y)

LogisticRegression(C=1.0, class_weight=None, dual=False, fit_intercept=True,
                   intercept_scaling=1, l1_ratio=None, max_iter=100,
                   multi_class='warn', n_jobs=None, penalty='l2',
                   random_state=None, solver='warn', tol=0.0001, verbose=0,
                   warm_start=False)

In [28]:
print("Train accuracy score:- ",vector_model.score(train_X,train_Y))
print("Test accuracy score:- ",vector_model.score(test_X,test_Y))

Train accuracy score:-  0.9827204253433761
Test accuracy score:-  0.8868175765645806
